In [1]:
import nest_asyncio
import os
import pickle
from llama_index.core import VectorStoreIndex

In [2]:
with open("embeddings.pkl", "rb") as f:
    embeds = pickle.load(f)

In [2]:
from pinecone.grpc import PineconeGRPC
from llama_index.vector_stores.pinecone import PineconeVectorStore
from llama_index.core import StorageContext

In [3]:
from dotenv import load_dotenv
from pinecone import ServerlessSpec
load_dotenv()
api = os.getenv("PINECONE_API_KEY")

pine = PineconeGRPC(api_key=api)


In [ ]:
pine.create_index("embeddings", dimension=1024, metric='cosine', spec=ServerlessSpec(cloud="aws",region="us-east-1"))

In [23]:
with open("extracts.pkl", "rb") as f:
    nodes = pickle.load(f)

In [8]:
index = pine.Index("embeddings")

In [9]:
vectorstore = PineconeVectorStore(pinecone_index=index)

In [27]:
vectors=[]
for i, embed in enumerate(embeds):
    nodal = {
        "id" : f"{i}",
        "values": embed,
        "metadata": { "text": nodes[i].text, **nodes[i].metadata}
        
    }
    vectors.append(nodal)


In [32]:
for i in range(0,len(vectors), 100):
    batch = vectors[i:i+100]
    if i == 9500:
        batch = vectors[i:]
    index.upsert(vectors=batch)
    print("upserted ",i)

upserted  0
upserted  100
upserted  200
upserted  300
upserted  400
upserted  500
upserted  600
upserted  700
upserted  800
upserted  900
upserted  1000
upserted  1100
upserted  1200
upserted  1300
upserted  1400
upserted  1500
upserted  1600
upserted  1700
upserted  1800
upserted  1900
upserted  2000
upserted  2100
upserted  2200
upserted  2300
upserted  2400
upserted  2500
upserted  2600
upserted  2700
upserted  2800
upserted  2900
upserted  3000
upserted  3100
upserted  3200
upserted  3300
upserted  3400
upserted  3500
upserted  3600
upserted  3700
upserted  3800
upserted  3900
upserted  4000
upserted  4100
upserted  4200
upserted  4300
upserted  4400
upserted  4500
upserted  4600
upserted  4700
upserted  4800
upserted  4900
upserted  5000
upserted  5100
upserted  5200
upserted  5300
upserted  5400
upserted  5500
upserted  5600
upserted  5700
upserted  5800
upserted  5900
upserted  6000
upserted  6100
upserted  6200
upserted  6300
upserted  6400
upserted  6500
upserted  6600
upserte

In [4]:
from llama_index.llms.groq import Groq
load_dotenv()

True

In [5]:
api = os.getenv("GROQ_API_KEY")
llm = Groq(model="openai/gpt-oss-120b", api=api)

In [6]:
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core import get_response_synthesizer

In [10]:
from llama_index.core import Settings
Settings.llm = llm

from llama_index.embeddings.huggingface_api import HuggingFaceInferenceAPIEmbedding
import nest_asyncio

nest_asyncio.apply()
embed_model = HuggingFaceInferenceAPIEmbedding(
    model_name="BAAI/bge-m3",
    token=os.getenv("HF_API_KEY"),
)
Settings.embed_model=embed_model

vectorindex = VectorStoreIndex.from_vector_store(vector_store=vectorstore, llm=llm)



In [16]:


retriver = VectorIndexRetriever(index=vectorindex,verbose=True, top_k=20)
synth = get_response_synthesizer()



In [17]:
from llama_index.core.postprocessor import SentenceTransformerRerank

rerank = SentenceTransformerRerank(model="cross-encoder/ms-marco-MiniLM-L4-v2", top_n=7)

query = input("Enter your query: ")

prompt_instructions = f'You are a part of a conservation research RAG assistant - improve the prompt for the LLM to yield better results. The user has entered the prompt below:\n {query} \n The prompt could have limitations such as being too specific where the corpus may not have information on the exact species or too generalised for a corpus detailing technical insights into species conservation. It may also not provide constraints and rules to the LLM \n Your job is to enhance the query so that RAG retrieval returns interesting points of discussion. Ensure you mention that all information must be backed by citations sourced only from the actual RAG vector database, tell it to recognise levels of uncertainty of information due to the corpus and a breakdown of information reasoning which does not go beyond species conservation. \n in Ensure the new prompt is between 50-60 words max. \nOnly return the prompt text.'

new_prompt = llm.complete(prompt_instructions).text
print(new_prompt)

query_engine = RetrieverQueryEngine(retriever=retriver, response_synthesizer=synth, node_postprocessors=[rerank])
response = query_engine.query(new_prompt)
nodes = response.source_nodes

print([node.score for node in nodes])

Provide a concise overview of white‑tiger conservation, including current threats, population trends, and effective management strategies. Cite all statements using only sources retrieved from the RAG vector database, indicate any uncertainty levels, and present a brief reasoning breakdown limited to species‑conservation aspects. Discuss genetic health, habitat connectivity, and community involvement.
[-3.448951, -7.8897305]


In [20]:
print([node.text for node in nodes])

['Historical distributions are notoriously difficult to establish and can probably never be \ndefined with certainty.  When mapping historical distributions, the planning team should \nconsider historical records (taking into account the records’ provenance and assessed \nreliability), known habitat affinities, and changes in the human landscape (e.g., human \npopulation density, land-use, etc.) over time.  Source material and any assumptions should \nbe carefully documented.  The map itself should be prepared as a GIS layer and \ndocumented with appropriate metadata.  An example of a historical distribution map for \ntigers is shown in Figure 5.1.  This \nmap shows that tigers now occupy \n7% of their historic range. \n \n \nFor some species, historical \ninformation related to population size \nis also available.  Ideally some \nmeasure of certainty and quality \nshould be attached to these data.  \nFor example, hunting harvests, \nsighting records, or rough counts \nfrom the past mi

In [18]:
print(response)

**White‑tiger conservation – concise overview**

| Aspect | Summary | Uncertainty* | Source |
|--------|---------|--------------|--------|
| **Current threats** | • Ongoing habitat loss and fragmentation continue to shrink the area that tigers can occupy.<br>• Poaching for body parts and illegal trade remain major mortality factors.<br>• Small, isolated populations are vulnerable to stochastic events and inbreeding. | High – the excerpt discusses general tiger threats only indirectly; specific data for white‑tigers are not provided. | – |
| **Population trend** | Tigers now occupy roughly **7 % of their historic range**, indicating a severe contraction that likely includes white‑tiger sub‑populations. | Medium – the 7 % figure is documented for the species as a whole; it is reasonable to infer a similar trend for the white‑tiger morph, but the exact proportion for that morph is unknown. | Figure 5.1 (tiger historic vs. present range) |
| **Effective management strategies** | • **Standa

In [19]:
query_engine = RetrieverQueryEngine(retriever=retriver, response_synthesizer=synth)
query = input("Enter your query: ")
response = query_engine.query(new_prompt)
print(response)

**White‑tiger conservation – concise overview**

| Aspect | Summary | Uncertainty |
|--------|---------|-------------|
| **Current threats** | Ongoing loss and fragmentation of tiger habitat driven by expanding human populations and land‑use change continue to limit the areas where tigers can persist. | Moderate – the excerpt notes that “changes in the human landscape (e.g., human population density, land‑use, etc.)” must be considered when mapping historic distributions, implying these pressures remain relevant today. |
| **Population trends** | Tigers now occupy roughly **7 %** of their historic range, indicating a severe contraction in distribution and a likely decline in overall numbers. | Low – the figure is explicitly reported. |
| **Genetic health** | White‑tigers are a colour morph of the species; without specific genetic monitoring data, the status of their genetic diversity cannot be quantified. | High – no direct data on genetic health are provided in the sources. |
| **Habi

In [ ]:

"""
schema for design: 

user enters: conservation techniques for this and that

rephrase with prompt: (query expansion)
provide a sophisticated analysis of the point of disucssion. Follow the schema provided to generate a thoughtful report of the question based on the corpus. Ensure that all claims are referenced with page number and file name. Draw conclusions based on the information from the corpus. if you cannot find any reasonable link with the corpus, make it clear that there are gaps in place. 

schema: 
{
    'Introduction' : pass
    'Key insights and strategies':
    'Confidence score /10':
    'List of statistics':
    'References':
}

new prompt is taken and entered into the vector db. Results returned. 

Query stratgies - reranking, 

"""